<a href="https://colab.research.google.com/github/speculaas/ai201-project3-takemeter/blob/main/fable5_traces_explorer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fable-5 Traces Explorer
### AI201 · Project 3 · TakeMeter

Lightweight Colab notebook for **inspecting, sampling, flattening, and exporting** readable subsets of the [Glint-Research/Fable-5-traces](https://huggingface.co/datasets/Glint-Research/Fable-5-traces) dataset.

**Important schema note:** `load_dataset(...)` exposes the `pi_agent` config. Each row is one **trace event** (`session`, `model_change`, `thinking_level_change`, `message`, …), not a whole session with top-level `messages` / `tools` / `trace` fields. That is why `.get("session_id")` and similar lookups return `None`.

Actual event fields: `type`, `id`, `timestamp`, `parentId`, `modelId`, `thinkingLevel`, `cwd`, `message`, `version`.

The repo also ships `fable5_cot_merged.jsonl` (flat SFT rows: `uid`, `session`, `context`, `cot`, `output_type`, …). See the optional section near the bottom if you want that view.

---
**Before you start:**
- Use a **CPU** runtime: Runtime → Change runtime type → Hardware accelerator: **None**
- Opening or refreshing the notebook page is cheap; **connected runtimes and running code** are what consume quota
- When finished: Runtime → **Disconnect and delete runtime**
- Save exports to Drive or download them; runtime files disappear when the VM dies

## Colab runtime vs terminal

Colab has two layers:
1. **Notebook UI** — the `.ipynb` in your browser
2. **Runtime VM** — the temporary Linux machine where Python, shell commands, Git, files, and installs live

The Colab **terminal** runs in the **same runtime** as the notebook. They share:
- the same filesystem (e.g. `/content`)
- the same `pip` / `apt` installs
- the same runtime lifetime (when the runtime dies, terminal state dies too)

Important caveats:
- **`export VAR=...` in terminal** does not reach the already-running Python kernel. Set env vars in Python (`os.environ[...]`) or write a config file both sides can read.
- **`cd` in terminal** does not change the notebook kernel cwd. In notebook cells use `%cd /content/path`, not `!cd`.
- **Put reproducible setup in notebook cells** (`!pip install ...`) so Runtime → Restart runtime + Run all recreates the environment.
- **Use terminal for quick git/file debugging**; use notebook cells for installs, data work, and exports.

Opening another notebook tab is mostly UI until you **Connect** or run a cell. Avoid keeping several connected runtimes open at once.

## Setup: refresh repo + install deps

Run this first after Cursor pushes fixes to GitHub. If you opened the notebook directly from GitHub (Open in Colab badge), this clones the repo into `/content` so terminal and notebook share the same checkout.

In [ ]:
import os
from pathlib import Path

REPO_URL = "https://github.com/speculaas/ai201-project3-takemeter.git"
REPO_DIR = "/content/ai201-project3-takemeter"
BRANCH = "main"

if not Path(REPO_DIR).exists():
    !git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git fetch origin {BRANCH}
    !git reset --hard origin/{BRANCH}

%cd {REPO_DIR}
!git status --short --branch

!python -m pip install -q datasets pandas pyarrow huggingface_hub

from datasets import load_dataset
import json
import pandas as pd
from itertools import islice
from collections import Counter

print("✅ Repo ready:", REPO_DIR)
print("✅ Dependencies ready")

## Helpers

In [ ]:
def preview(value, n=1000):
    if isinstance(value, (dict, list)):
        s = json.dumps(value, indent=2, ensure_ascii=False)
    else:
        s = str(value)
    return s[:n]

def short(x, n=500):
    if x is None:
        return None
    if isinstance(x, (dict, list)):
        x = json.dumps(x, ensure_ascii=False)
    else:
        x = str(x)
    return x[:n]

def flatten_event(i, r):
    msg = r.get("message") or {}
    role = msg.get("role") if isinstance(msg, dict) else None
    content = msg.get("content") if isinstance(msg, dict) else None
    content_types = None
    if isinstance(content, list):
        content_types = ",".join(
            c.get("type", "?") for c in content if isinstance(c, dict)
        )
    return {
        "i": i,
        "type": r.get("type"),
        "id": r.get("id"),
        "timestamp": r.get("timestamp"),
        "parentId": r.get("parentId"),
        "modelId": r.get("modelId"),
        "thinkingLevel": r.get("thinkingLevel"),
        "cwd": r.get("cwd"),
        "message_role": role,
        "content_types": content_types,
        "message_preview": short(msg, 500),
    }

def group_events_into_sessions(events):
    sessions = []
    current = None
    for e in events:
        if e.get("type") == "session":
            current = {
                "session_id": e.get("id"),
                "cwd": e.get("cwd"),
                "events": [e],
            }
            sessions.append(current)
        elif current is not None:
            current["events"].append(e)
    return sessions

def session_summary(session):
    events = session["events"]
    model_ids = [e.get("modelId") for e in events if e.get("modelId")]
    thinking_levels = [e.get("thinkingLevel") for e in events if e.get("thinkingLevel")]
    messages = [e for e in events if e.get("type") == "message"]
    return {
        "session_id": session.get("session_id"),
        "cwd": session.get("cwd"),
        "num_events": len(events),
        "num_messages": len(messages),
        "modelId": model_ids[0] if model_ids else None,
        "thinkingLevel": thinking_levels[-1] if thinking_levels else None,
    }

## Stream the first few rows

Hugging Face `datasets` supports `streaming=True`. The only published config is `pi_agent`.

In [ ]:
ds = load_dataset(
    "Glint-Research/Fable-5-traces",
    "pi_agent",
    split="train",
    streaming=True,
)

rows = list(islice(ds, 40))
print("keys:", sorted(rows[0].keys()))
print("first 10 event types:", [r.get("type") for r in rows[:10]])

## Inspect one row compactly

In [ ]:
row = rows[0]
for k, v in row.items():
    print("=" * 80)
    print(k, type(v))
    print(preview(v, 1500))

## Flatten core fields into a readable table

This table is **event-level**. For session-level summaries, run the next cell.

In [ ]:
records = [flatten_event(i, r) for i, r in enumerate(rows)]
df = pd.DataFrame(records)
df

## Group events into sessions

In [ ]:
sessions = group_events_into_sessions(rows)
session_df = pd.DataFrame(session_summary(s) for s in sessions)
print(f"found {len(sessions)} sessions in first {len(rows)} events")
session_df

## Inspect assistant message blocks

Assistant turns usually contain `thinking`, `text`, and/or `toolCall` blocks inside `message.content`.

In [ ]:
assistant_row = next(
    r for r in rows
    if r.get("type") == "message"
    and isinstance(r.get("message"), dict)
    and r["message"].get("role") == "assistant"
)
content = assistant_row["message"].get("content", [])
print("content block types:", [c.get("type") for c in content if isinstance(c, dict)])
preview(content, 3000)

## Show event types

In [ ]:
event_types = Counter(r.get("type", "_missing") for r in rows)
event_types

## Export a readable Markdown file

Files land in the repo checkout under `/content/...`. Download them or copy to Drive before disconnecting the runtime.

In [ ]:
def session_to_markdown(session, index=0):
    md = []
    md.append(f"# Fable-5 Session Example {index}")
    md.append("")
    md.append(f"- **session_id**: `{session.get('session_id')}`")
    md.append(f"- **cwd**: `{session.get('cwd')}`")
    md.append(f"- **num_events**: `{len(session.get('events', []))}`")
    md.append("\n## Session summary\n")
    md.append("```json")
    md.append(json.dumps(session_summary(session), indent=2, ensure_ascii=False))
    md.append("```")
    md.append("\n## Events preview\n")
    md.append("```json")
    md.append(json.dumps(session.get("events", [])[:20], indent=2, ensure_ascii=False))
    md.append("```")
    return "\n".join(md)

OUT_DIR = Path(REPO_DIR)
markdown = session_to_markdown(sessions[0], 0)
out_path = OUT_DIR / "fable5_session_0.md"
out_path.write_text(markdown, encoding="utf-8")
print("wrote", out_path)

## Export N events as JSONL

Adjust `N` as needed. Each output row is one Pi trace event.

In [ ]:
ds = load_dataset(
    "Glint-Research/Fable-5-traces",
    "pi_agent",
    split="train",
    streaming=True,
)

N = 100
out_path = Path(REPO_DIR) / f"fable5_first_{N}_events.jsonl"
with open(out_path, "w", encoding="utf-8") as f:
    for i, r in enumerate(islice(ds, N)):
        out = {"i": i, **r}
        f.write(json.dumps(out, ensure_ascii=False) + "\n")

print("wrote", out_path)

## What to look for in Fable-5 rows

Search nested structures for thinking blocks, tool calls, model changes, and message roles.

In [ ]:
interesting_keys = [
    "thinking", "reasoning", "reasoning_content",
    "toolCall", "tool_use", "tool_result",
    "modelId", "thinkingLevel", "message", "role", "content",
]

def search_obj(obj, needle, path=""):
    hits = []
    if isinstance(obj, dict):
        for k, v in obj.items():
            p = f"{path}.{k}" if path else k
            if needle.lower() in str(k).lower():
                hits.append((p, v))
            hits.extend(search_obj(v, needle, p))
    elif isinstance(obj, list):
        for i, v in enumerate(obj):
            hits.extend(search_obj(v, needle, f"{path}[{i}]"))
    elif needle.lower() in str(obj).lower():
        hits.append((path, obj))
    return hits

for needle in interesting_keys:
    hits = search_obj(row, needle)
    if hits:
        print("\n###", needle)
        for p, v in hits[:10]:
            print(p, "=>", str(v)[:300])

## Optional: inspect flat `fable5_cot_merged.jsonl` rows

This is the merged SFT file shown in the dataset README / HF viewer (`uid`, `session`, `context`, `cot`, `output_type`, `output`, `completion`, `origin`). It is separate from the `pi_agent` event stream above.

In [ ]:
from huggingface_hub import hf_hub_download

merged_path = hf_hub_download(
    repo_id="Glint-Research/Fable-5-traces",
    filename="fable5_cot_merged.jsonl",
    repo_type="dataset",
)

merged_rows = []
with open(merged_path, "r", encoding="utf-8") as f:
    for line in islice(f, 5):
        merged_rows.append(json.loads(line))

print("merged keys:", sorted(merged_rows[0].keys()))
pd.DataFrame([
    {
        "uid": r.get("uid"),
        "session": r.get("session"),
        "model": r.get("model"),
        "output_type": r.get("output_type"),
        "origin": r.get("origin"),
        "context_preview": short(r.get("context"), 200),
        "cot_preview": short(r.get("cot"), 200),
    }
    for r in merged_rows
])

## Bottom line

If your flatten table was all `null`, you were probably using the wrong field names. The HF `pi_agent` split stores **one event per row**, not one session per row.

Use **streaming**, export small sampled subsets, and save artifacts to Drive. When reverse-engineering traces, look inside `message.content` for `thinking`, `text`, and `toolCall` blocks, and use session grouping when you need whole-conversation context.

**Colab workflow:** Cursor edits on GitHub → rerun the setup cell here → explore. Use **CPU**, disconnect the runtime when done, and don't rely on terminal-only setup that won't survive a restart.